# Problem Statement

Machine learning algorithms do not work if the given data has missing values. Removing missing values is called imputation. This process must be done before an ML model can be trained. 

However, not all algorithms have a "hatred" for missing values. Some algorithms can handle missing values. Xgboost, for example, can handle missing values.
The goal is to make a feature imputor which used the `xgboost` algorithm to impute missing values.

# How Can This be Done?

The basic idea is to use the Xgboost algorithm to predict the missing values in a particular column by using the values in the other columns. There are quite a few steps involved for this to work. Let's do it step by step.

# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Steps

In [134]:
df = pd.read_csv('data/train.csv')

In [135]:
columns = df.columns
COL = "HomePlanet"
original_target = df[COL]

## Storing some information

In [136]:
num_unique_values = {}
for col in columns:
    num_unique_values[col] = df[col].nunique()

col_dtypes = df.dtypes

## Creating `to_train` and `to_predict` dataframes

In [137]:
df_to_train = df[df[COL].notna()]
df_to_predict = df[df[COL].isna()]
target = df_to_train[COL]
len(df_to_train), len(df_to_predict)

(8492, 201)

## Determining Which Columns to Drop

In [138]:
cols_to_drop = [COL]
max_unique = 10
for col in columns:
    col_is_object = col_dtypes[col] == "object"
    if ((num_unique_values[col] > max_unique) and col_is_object):
        cols_to_drop.append(col)
cols_to_drop

['HomePlanet', 'PassengerId', 'Cabin', 'Name']

In [139]:
df_to_train = df_to_train.drop(cols_to_drop, axis=1)
df_to_predict = df_to_predict.drop(cols_to_drop, axis=1)
df_to_predict_index = df_to_predict.index.values

## One Hot Encoding

In [140]:
df_to_train = pd.get_dummies(df_to_train, drop_first=True)
df_to_predict = pd.get_dummies(df_to_predict, drop_first=True)

In [141]:
df_to_train.columns

Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Transported', 'CryoSleep_True', 'Destination_PSO J318.5-22',
       'Destination_TRAPPIST-1e', 'VIP_True'],
      dtype='object')

## Training the Model

In [142]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
target = le.fit_transform(target)

In [143]:
xgb_model = xgb.XGBClassifier(n_estimators=int(np.sqrt(len(df_to_train))), max_depth=5, learning_rate=0.1)
xgb_model.fit(df_to_train, target)

XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
              importance_type=None, interaction_constraints='',
              learning_rate=0.1, max_bin=256, max_cat_to_onehot=4,
              max_delta_step=0, max_depth=5, max_leaves=0, min_child_weight=1,
              missing=nan, monotone_constraints='()', n_estimators=92, n_jobs=0,
              num_parallel_tree=1, objective='multi:softprob', predictor='auto',
              random_state=0, reg_alpha=0, ...)

## Predicting the Missing Values

In [144]:
xgb_model.score(df_to_train, target)

0.8002826189354687

In [145]:
preds = xgb_model.predict(df_to_predict)
preds = le.inverse_transform(preds)
preds

array(['Europa', 'Europa', 'Europa', 'Earth', 'Europa', 'Earth', 'Earth',
       'Mars', 'Earth', 'Europa', 'Mars', 'Earth', 'Europa', 'Europa',
       'Europa', 'Europa', 'Earth', 'Mars', 'Earth', 'Mars', 'Europa',
       'Europa', 'Mars', 'Earth', 'Earth', 'Europa', 'Europa', 'Earth',
       'Europa', 'Earth', 'Earth', 'Earth', 'Earth', 'Earth', 'Earth',
       'Earth', 'Earth', 'Earth', 'Earth', 'Earth', 'Earth', 'Earth',
       'Earth', 'Earth', 'Earth', 'Earth', 'Europa', 'Europa', 'Mars',
       'Earth', 'Earth', 'Earth', 'Europa', 'Mars', 'Earth', 'Earth',
       'Earth', 'Europa', 'Earth', 'Earth', 'Mars', 'Earth', 'Earth',
       'Earth', 'Earth', 'Earth', 'Earth', 'Earth', 'Earth', 'Earth',
       'Earth', 'Earth', 'Earth', 'Europa', 'Earth', 'Earth', 'Earth',
       'Earth', 'Mars', 'Europa', 'Europa', 'Earth', 'Earth', 'Europa',
       'Earth', 'Earth', 'Earth', 'Europa', 'Europa', 'Europa', 'Europa',
       'Earth', 'Earth', 'Earth', 'Earth', 'Earth', 'Europa', 'Earth',
  

## Merging this to the original dataframe

In [155]:
original_target_copy = original_target.copy()
original_target_copy.iloc[df_to_predict_index] = preds

In [156]:
original_target_copy.isna().sum()

0

In [157]:
df[COL] = original_target_copy
df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


# The `FeatureImputor` Class

In [6]:

class FeatureImputor:
    def __init__(self, df, features=None, max_unique_to_drop=15):
        self.df = self._read_data(df)

        self.features = features
        self.length = len(self.df)
        self.max_unique_to_drop = max_unique_to_drop
        self.all_features = self.df.columns
        if features is None:
            self.features = self.all_features
        self._some_details()
        self.model = None
        self.le = LabelEncoder()
    def _read_data(self, df):
        if isinstance(df, str):
            return pd.read_csv(df)
        elif isinstance(df, pd.DataFrame):
            return df

    def _some_details(self):
        num_unique_values = {}
        missing_values = {}
        for feature in self.all_features:
            num_unique_values[feature] = self.df[feature].nunique()
            missing_values[feature] = self.df[feature].isna().sum()
        self.num_unique_values = num_unique_values
        self.missing_values = missing_values
        self.col_dtypes = self.df.dtypes

    def _drop_features(self, col):
        cols_to_drop = [col]
        max_unique = self.max_unique_to_drop
        for column in self.all_features:
            col_is_object = self.col_dtypes[column] == "object"
            if (self.num_unique_values[column] > max_unique) and col_is_object:
                cols_to_drop.append(column)
        return cols_to_drop

    def _create_train_pred_dataset(self, col):
        cols_to_drop = self._drop_features(col)
        original_target = self.df[col]
        df_to_train = self.df[self.df[col].notna()]
        df_to_predict = self.df[self.df[col].isna()]
        target = df_to_train[col]
       
        df_to_train = df_to_train.drop(cols_to_drop, axis=1)
        df_to_predict = df_to_predict.drop(cols_to_drop, axis=1)
        df_to_predict_index = df_to_predict.index.values
        self.df_to_train = df_to_train
        self.df_to_predict = df_to_predict
        self.df_to_predict_index = df_to_predict_index
        self.target = target
        self.original_target = original_target

    def _create_dummy(self):
        self.df_to_train = pd.get_dummies(self.df_to_train, drop_first=True)
        self.df_to_predict = pd.get_dummies(self.df_to_predict, drop_first=True)

    def _label_target(self, target):
        if self.col_dtypes[target]=="object":
            self.target = self.le.fit_transform(self.target)
        else:
            self.target = self.target
        # return self.target
    def _check_columns(self):
        train_cols = self.df_to_train.columns
        predict_cols = self.df_to_predict.columns
        # if len(train_cols) != len(predict_cols):
            # print("Colimns Mismatch. Adjusting...")
        if len(train_cols) > len(predict_cols):
            self.df_to_predict = self.df_to_predict.reindex(columns=train_cols)
        elif len(train_cols) < len(predict_cols):
            self.df_to_train = self.df_to_train.reindex(columns=predict_cols)

    def _predict(self, target):
        dtype = self.col_dtypes[target]
        self._check_columns()
        if dtype != "object":
            # print("Using XGBoostRegressor")
            self.model = xgb.XGBRFRegressor(n_estimators=100, max_depth=5, learning_rate=0.1)
            decode=False
        else:
            # print("Using XGBoostClassifier")
            self.model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
            decode=True

        self.model.fit(self.df_to_train, self.target)
        # print("Fitted!")
        preds = self.model.predict(self.df_to_predict)
        if decode:
            preds = self.le.inverse_transform(preds)
        self.preds = preds
        return preds

    def _fill_missing_values(self, col):
        original_target_copy = self.original_target.copy()
        original_target_copy.iloc[self.df_to_predict_index] = self.preds
        self.df[col] = original_target_copy
        return self.df
    
    def _impute_one(self, col):
        print(f"Filling {self.missing_values[col]} missing values in {col}!")
        self._create_train_pred_dataset(col=col)
        # print("Creating dummy variables")
        self._create_dummy()
        # print("Labeling target")
        self._label_target(col)
        # print("Predicting")
        self._predict(col)
        # print("Filling missing values")
        self._fill_missing_values(col)
        
        return self.df
    
    def impute(self, file_path = ""):
        cols = self.features
        for col in cols:
            if self.missing_values[col] <= 0:
                continue
            if (self.max_unique_to_drop < self.num_unique_values[col]) and (self.col_dtypes[col] == "object"):
                print("Column: ", col, " has too many unique values. Skipping...")
                continue
            self.df = self._impute_one(col)
        if file_path:
            self.df.to_csv(file_path, index=False)
        return self.df

In [7]:
imputor = FeatureImputor('data/train_cabin.csv')
df2 = imputor.impute("train_clean.csv")

Filling 201 missing values in HomePlanet!
Filling 217 missing values in CryoSleep!
Filling 182 missing values in Destination!
Filling 179 missing values in Age!
Filling 203 missing values in VIP!
Filling 181 missing values in RoomService!
Filling 183 missing values in FoodCourt!
Filling 208 missing values in ShoppingMall!
Filling 183 missing values in Spa!
Filling 188 missing values in VRDeck!
Column:  Name  has too many unique values. Skipping...
Filling 199 missing values in CabinFirst!
Filling 199 missing values in CabinLast!


In [8]:
df2.isna().sum()

PassengerId       0
HomePlanet        0
CryoSleep         0
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            200
Transported       0
CabinFirst        0
CabinLast         0
Cabin_Num         0
dtype: int64

In [10]:
imputor = FeatureImputor('data/test_cabin.csv')
df3 = imputor.impute("test_clean.csv")

Filling 87 missing values in HomePlanet!
Filling 93 missing values in CryoSleep!
Filling 92 missing values in Destination!
Filling 91 missing values in Age!
Filling 93 missing values in VIP!
Filling 82 missing values in RoomService!
Filling 106 missing values in FoodCourt!
Filling 98 missing values in ShoppingMall!
Filling 101 missing values in Spa!
Filling 80 missing values in VRDeck!
Column:  Name  has too many unique values. Skipping...
Filling 100 missing values in CabinFirst!
Filling 100 missing values in CabinLast!


In [11]:
df3.isna().sum()

PassengerId      0
HomePlanet       0
CryoSleep        0
Destination      0
Age              0
VIP              0
RoomService      0
FoodCourt        0
ShoppingMall     0
Spa              0
VRDeck           0
Name            94
CabinFirst       0
CabinLast        0
Cabin_Num        0
dtype: int64